In [4]:

import os
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec
from scipy.interpolate import Rbf
import umap

# Add project root to path
sys.path.append(os.path.abspath('../../'))
from BWM.ScriptsandFunctions.bwm_functions import *

# Set global plotting parameters
plt.rcParams["axes.grid"] = False
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 7,
    'axes.titlesize': 7,
    'axes.labelsize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 7,
    'figure.titlesize': 7
})


In [1]:
# Helper functions
def iso_ae_preprocessing_tensor(response_tensor):
    # Z-score and reshape
    response_tensor_norm = (response_tensor - np.mean(response_tensor, axis=(0, 1), keepdims=True)) / (np.std(response_tensor, axis=(0, 1), keepdims=True) + 1e-8)
    # response_tensor_norm shape: (conditions, time, neurons)
    # Reshape to (conditions * time, neurons) and then transpose to (neurons, features)
    response_matrix = response_tensor_norm.reshape(-1, response_tensor_norm.shape[2])
    X_transposed = response_matrix.T

    # Scale each row (neuron) to unit length
    row_norms = np.linalg.norm(X_transposed, axis=1, keepdims=True)
    X_pre_pca = X_transposed / (row_norms + 1e-8)

    return X_pre_pca

def calculate_msd_data(response_tensor):
    """
    Calculate Mean Standardized Derivative using finite differences.
    responses: (conditions, time, neurons)
    """
    z = (response_tensor - np.mean(response_tensor, axis=(0, 1), keepdims=True)) / (np.std(response_tensor, axis=(0, 1), keepdims=True) + 1e-8)
    speed = np.mean(np.abs(np.diff(z, axis=1)), axis=(0, 1))
    return speed


In [2]:
# Configuration
directory_path = '../Data2'
event = 'stimOn_times'
conditions = ['choice']


In [5]:
eid = '6274dda8-3a59-4aa1-95f8-a8a549c46a26'
res = get_eid_response_matrix(
        eid, directory_path, event, conditions,
        n_pseudo=25, threshold=None, std_above=50, std_below=50, start_time=0,
        filter_significant=False, n_steps=100, step=0.005, stride=0.005, sigma=2
    )


pseudo_response_tensor, _, beryl_regs, _, _ = res

In [ ]:
# Plot densmap projection of EID responses
#Figure 1
fig = plt.figure(figsize=(6, 2))
gs = gridspec.GridSpec(1, 3)
ax1 = plt.subplot(gs[0,0])
ax2 = plt.subplot(gs[0,1])
ax3 = plt.subplot(gs[0,2])
# First row
#from sklearn.manifold import Isomap
z_neurons = umap.UMAP(densmap = True, dens_lambda=2, n_neighbors=500).fit_transform(X_input)
sns.scatterplot(x = z_neurons[:,0], y= z_neurons[:,1], color = 'gray',ax=ax1,s=1,legend=False,alpha =.5)
sns.scatterplot(x = z_neurons[:,0], y= z_neurons[:,1], hue = msd_data,ax=ax2,palette='turbo',s=1,alpha = .5,legend=False)

from scipy.interpolate import Rbf

# Calculate limits for plotting based on the initial state
x_min, x_max = z_neurons[:,0].min(), z_neurons[:,0].max()
y_min, y_max = z_neurons[:,1].min(), z_neurons[:,1].max()
pad = 0.5

# Prepare RBF background
grid_x, grid_y = np.mgrid[x_min-pad:x_max+pad:100j, y_min-pad:y_max+pad:100j]
rbf = Rbf(z_neurons[:,0], z_neurons[:,1], msd_data, function='multiquadric', smooth=1)
grid_msd = rbf(grid_x, grid_y)
sns.heatmap(grid_msd.T,ax=ax3,cmap='Spectral_r')
